# TP2 - Informe técnico

Sistema de Detección y Clasificación de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1: Balverdi, Valentina
- Alumno 2: Prieto, Tobías

## 1. Explicación completa del pipeline

El sistema implementa de forma incremental el flujo **Embeddings → Búsqueda por similitud → Clasificación → Detección → Pipeline completo**, siguiendo las tres etapas pedidas por la cátedra. Cada etapa se desacopla en un servicio independiente del repositorio base, lo que facilita testing, debugging y extensibilidad.

**Etapa 1 — Buscador por similitud (`similarity_service.py`)**

1. Se toma una imagen de consulta en formato BGR (OpenCV).
2. `extract_embedding(image)` convierte la imagen en un vector de 768 dimensiones utilizando **ConvNeXt-Tiny** pre-entrenado en ImageNet sin la última capa de clasificación (`classifier[2] = Identity()`).
3. `search_similar_images(embedding, top_k)` recupera los `top_k` vecinos más cercanos desde **PostgreSQL + pgvector**, utilizando coseno (`<=>`) o distancia L2 (`<->`) según la configuración. La base se indexa una sola vez sobre el split de train balanceado.
4. `predict_breed_from_neighbors(results)` agrega los vecinos por **voto ponderado por similitud**, devolviendo `"unknown"` cuando el mejor vecino no supera el `SIMILARITY_THRESHOLD` (0.55). Para L2 la distancia se convierte a similitud con `1 / (1 + d)` para usar la misma lógica de voto.

**Etapa 2 — Clasificación supervisada (`classifier_service.py`)**

1. `train_classifier()` entrena el modelo activo sobre `data/dataset_balanced/train` y guarda el checkpoint en la ruta configurada.
2. Se entrenan dos modelos comparables: **ResNet18 fine-tuned** (Modelo A obligatorio) y una **CNN custom** (Modelo B, propia).
3. `evaluate_classifier()` evalúa el modelo activo sobre el split de test y devuelve accuracy, precision, recall, specificity y F1.
4. La aplicación Gradio permite seleccionar dinámicamente el modelo activo mediante variables de entorno (`CLASSIFIER_ACTIVE_MODEL`).

**Etapa 3 — Detección + clasificación end-to-end (`detection_service.py`)**

1. `detect_dogs(image)` usa **YOLO11s** pre-entrenado en COCO, filtrando únicamente la clase `dog` (id 16). Retorna `[((x1, y1, x2, y2), conf), ...]`.
2. Para cada bounding box se recorta la región de la imagen original.
3. `classify_detected_dog(crop)` aplica un preprocesamiento con **SquarePad** (para no deformar los recortes rectangulares) y luego pasa la imagen por el ResNet18 fine-tuneado de la Etapa 2, devolviendo `(raza, score)`.
4. La aplicación Gradio dibuja sobre la imagen original las bounding boxes, las etiquetas de raza y los scores de confianza tanto de detección como de clasificación.

**Infraestructura.** Todo el sistema corre dentro de Docker Compose: un servicio PostgreSQL+pgvector para la base vectorial y un servicio Python con la app Gradio. Las configuraciones (modelo activo, threshold, top-k, paths) se inyectan vía variables de entorno (`.env`), sin valores hardcodeados.

## 2. Dataset

Se utiliza el **70 Dog Breeds Image Dataset** de Kaggle, con **9.346 imágenes distribuidas en 70 razas** y tres splits provistos: `train`, `valid` y `test`.

### Distribución de clases por split

| Métrica | train | valid | test | total |
|---|---:|---:|---:|---:|
| Imágenes totales | 7.946 | 700 | 700 | 9.346 |
| Promedio por raza | 113,5 | 10 | 10 | 133,5 |
| Desvío estándar | 24,7 | 0 | 0 | 24,7 |
| Mínimo por raza | 65 (American Hairless) | 10 | 10 | 85 |
| Máximo por raza | 198 (Shih-Tzu) | 10 | 10 | 218 |

Los splits **valid** y **test** están perfectamente balanceados (exactamente 10 imágenes por raza). El split **train**, en cambio, presenta un desbalance moderado con un ratio max/min de **3,05×**: Shih-Tzu y Lhasa están por encima de 160 imágenes, mientras que American Hairless y Boxer están por debajo de 80. No es un desbalance extremo, pero sí suficiente como para sesgar el entrenamiento hacia las razas mayoritarias si no se corrige.

### Filtrado de imágenes de baja calidad

Se descartan las imágenes desenfocadas mediante la **varianza del Laplaciano** con un umbral conservador de 30. El operador de Laplace detecta cambios bruscos de intensidad: una imagen nítida tiene bordes bien definidos (alta varianza), una borrosa los tiene suavizados (baja varianza). Solo **5 imágenes** de las 7.946 de train fueron descartadas (0,06 %).

### Balanceo del split de entrenamiento

Para mitigar el desbalance se implementó `scripts/balance_train_dataset.py`, que genera una copia balanceada del split de train mediante **oversampling con data augmentation** sin tocar el dataset original. Cada clase queda en exactamente **200 imágenes**, totalizando **14.000 imágenes** en `data/dataset_balanced/train`. Las clases con déficit se completan con imágenes sintéticas generadas con `albumentations` (las aumentaciones se describen en la sección de preprocesamiento).

### Conjunto independiente de evaluación

Para validar la capacidad de generalización fuera del dataset de Kaggle se construyó un **dataset propio** (`data/dataset/propio`) con imágenes obtenidas de internet y fotografías propias. Contiene **70 imágenes** distribuidas en 7 razas (10 imágenes por raza): **Yorkie, Pomeranian, Dalmation, Rottweiler, Border Collie, Boxer y Chihuahua**. Sobre este conjunto se evalúa el modelo ganador de la Etapa 2 y se validan también las funciones de detección de la Etapa 3.

## 3. Preprocesamiento

El preprocesamiento se separa en tres pipelines según el uso:

### 3.1 Pipeline de embeddings (Etapa 1)

Para ConvNeXt-Tiny se usa **exactamente** el pipeline asociado a sus pesos pre-entrenados (`weights.transforms()`):

- **Resize a 232 px** (lado corto).
- **Center crop a 224×224**.
- **Normalización con media/desvío de ImageNet** (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`).

Usar el mismo preprocesamiento que durante el entrenamiento original del modelo es crítico: cualquier cambio (otro resize, otra normalización) degrada la calidad de los embeddings porque los filtros aprendidos esperan estadísticas específicas en la entrada.

### 3.2 Pipeline de entrenamiento (Etapa 2)

Como el dataset balanceado ya incluye imágenes con aumentaciones, en el `DataLoader` se eligen aumentaciones más **leves** para mejorar la generalización sin sobre-modificar las imágenes:

- `RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1))` — variación de encuadre acotada.
- `RandomHorizontalFlip(p=0.5)` — un perro espejado sigue siendo el mismo perro.
- `RandomRotation(degrees=10)` — leves variaciones de ángulo.
- `ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02)` — robustez a condiciones de iluminación.
- `GaussianBlur(kernel_size=3)` con `p=0.1` — simula fotos imperfectas sin degradar demasiado.
- `ToTensor` + normalización ImageNet.

Para **valid y test** se usa un pipeline determinístico: `Resize(256)` + `CenterCrop(224)` + normalización ImageNet. No hay augmentation porque queremos evaluar siempre sobre las mismas imágenes y mantener al perro centrado.

### 3.3 Pipeline de balanceo (oversampling sintético)

El script `balance_train_dataset.py` aplica aumentaciones con `albumentations` **más agresivas** que las del entrenamiento, porque su función es generar nuevas imágenes (no perturbar marginalmente cada batch):

- `HorizontalFlip`.
- `Affine` (escala, traslación, rotación ±15°, shear) con `p=0.8` — es la aumentación dominante porque simula variaciones de pose y encuadre, que es lo más frecuente en fotos reales.
- Brillo/contraste o variación de tono (HSV) — uno de los dos, no ambos, para no apilar perturbaciones de color.
- Blur leve o ruido gaussiano, con `p=0.15` baja — para simular condiciones de captura imperfectas sin degradar demasiado la imagen.

### 3.4 Pipeline de recortes de detección (Etapa 3)

Los recortes que devuelve YOLO son **rectangulares** (un perro parado es alto, uno acostado es ancho). Aplicar `Resize((224, 224))` directamente deformaría las proporciones del animal y degradaría la clasificación. Por eso se creó `SquarePad`: rellena con negro hasta volver la imagen cuadrada **antes** del resize, preservando las proporciones del perro. Luego se aplica `Resize((224, 224))`, `ToTensor` y la normalización ImageNet que espera ResNet18.

## 4. Justificación de los modelos elegidos

### Etapa 1 — ConvNeXt-Tiny (embeddings)

Se eligió **ConvNeXt-Tiny** pre-entrenado en ImageNet como extractor de embeddings:

- Es una CNN moderna (2022) que incorpora ideas de Vision Transformers manteniendo la eficiencia de las CNN convencionales. Logra accuracy competitiva con Swin Transformer en ImageNet con menor costo computacional.
- La variante **Tiny** ofrece un buen compromiso entre calidad de representaciones y velocidad de inferencia (**~28 M de parámetros vs. ~88 M** de ConvNeXt-Base), apta para indexar miles de imágenes en CPU sin tiempos prohibitivos.
- Se reemplaza `classifier[2]` por `nn.Identity()` para descartar la proyección a las 1000 clases de ImageNet y quedarnos con el **vector de 768 dimensiones** previo a la clasificación, que es lo que se usa como embedding.

### Etapa 2.A — ResNet18 fine-tuned (Modelo obligatorio)

Se eligió **ResNet18** sobre ResNet50 porque:

- Tiene **~11 M de parámetros** vs ~25 M de ResNet50: menor consumo de memoria y mayor velocidad de inferencia, importante para correr en CPU dentro del contenedor Docker.
- Los bloques residuales evitan el problema de degradación en redes profundas y permiten que el fine-tuning converja rápido (20 épocas alcanzan).
- Para 70 clases con ~200 imágenes/clase balanceadas, ResNet18 ya tiene capacidad más que suficiente. Subir a ResNet50 hubiera dado un margen marginal a costa de más del doble de cómputo.

### Etapa 2.B — CNN custom (Modelo propio)

Se diseñó una CNN secuencial con **5 bloques convolucionales** (32 → 64 → 128 → 256 → 512 canales), cada uno con BatchNorm, ReLU y MaxPool, seguida de `AdaptiveAvgPool2d((1,1))` y una cabeza FC (512 → 128 → 70) con Dropout(0.3) e inicialización Kaiming.

- Se eligió una arquitectura **estilo VGG simplificada** porque es el patrón clásico para aprender features jerárquicas desde cero.
- BatchNorm + Kaiming init estabilizan el entrenamiento con `lr=1e-3` desde la primera época.
- `AdaptiveAvgPool2d` evita una FC enorme y reduce overfitting frente a un Flatten denso.
- Sirve para mostrar **cuánto cuesta entrenar desde cero** vs aprovechar transferencia de aprendizaje.

### Etapa 3 — YOLO11s (detector)

Se eligió **YOLO11s** sobre YOLOv8n:

- YOLO es un detector single-stage que procesa la imagen en una sola pasada, apto para inferencia en tiempo real.
- La variante **small (~9,46 M parámetros)** dio mejor precisión que `yolov8n` en las pruebas sobre el dataset propio, manteniendo una inferencia rápida en CPU, lo que asegura que el sistema funcione dentro del contenedor Docker sin requerir GPU.
- La clase `dog` está ya entrenada en COCO con muchísimo soporte, por lo que el detector pre-entrenado funciona sin fine-tuning.

### Trade-offs

| Modelo | Parámetros | Precisión | Velocidad CPU | Memoria | Rol |
|---|---:|---|---|---|---|
| ConvNeXt-Tiny | ~28 M | Alta (baseline sin fine-tuning) | Media | Media | Embeddings |
| ResNet18 FT | ~11 M | Muy alta (0,954 test) | Alta | Baja | Clasificador principal |
| CNN custom | ~5 M | Aceptable (0,870 test) | Muy alta | Muy baja | Baseline propio |
| YOLO11s | ~9,46 M | Muy alta (clase `dog`) | Alta | Baja | Detector |

Donde más se gana es en la combinación: **YOLO11s detecta + ResNet18 FT clasifica**, lo que da un pipeline preciso y razonablemente rápido en CPU.

## 5. Proceso de entrenamiento e hiperparámetros

### 5.1 ResNet18 fine-tuned (Modelo A)

Se carga ResNet18 con pesos pre-entrenados en ImageNet y se reemplaza la última FC por `nn.Linear(512, 70)`. **Se entrena toda la red** (no se freezea el backbone) para que los filtros se adapten a las texturas y formas particulares de las razas.

| Hiperparámetro | Valor |
|---|---|
| Optimizador | AdamW |
| Learning rate inicial | 1e-4 |
| Weight decay | 1e-4 |
| Loss | CrossEntropyLoss |
| Scheduler | ReduceLROnPlateau (mode=min, factor=0.5, patience=3) |
| Batch size | 32 |
| Épocas | 20 |
| Selección | Mejor val accuracy |

**Curva de entrenamiento (extracto).** El modelo arranca con val_acc = 0,914 en la época 1 (la transferencia de ImageNet ya empuja arriba) y converge rápido a val_acc ≈ 0,95 entre las épocas 3-5. La mejor val_acc se alcanza en la época 20 (**0,9586**), con val_loss ~0,15 mientras que train_acc llega a ~0,998: hay un leve overfitting pero la val_loss se mantiene estable, por lo que no fue necesario cortar antes.

### 5.2 CNN custom (Modelo B)

| Hiperparámetro | Valor |
|---|---|
| Optimizador | Adam |
| Learning rate inicial | 1e-3 |
| Weight decay | 1e-4 |
| Loss | CrossEntropyLoss |
| Scheduler | ReduceLROnPlateau (mode=min, factor=0.5, patience=5, min_lr=1e-6) |
| Gradient clipping | `clip_grad_norm_(max_norm=1.0)` |
| Batch size | 32 |
| Épocas máx. | 500 |
| Early stopping | patience = 50 sobre val_loss |
| Selección | Mejores pesos según val_loss |

La CNN custom **entrena desde cero** con `lr=1e-3` (un orden más alto que ResNet18, porque no parte de pesos pre-entrenados). Necesita muchas más épocas para converger: arranca con val_acc < 0,1 y va creciendo gradualmente, llegando a val_acc ~0,87 al final. El gradient clipping protege contra explosiones de gradientes en las primeras épocas con BatchNorm + lr alto.

## 6. Resultados obtenidos

### 6.1 Etapa 1 — Búsqueda por similitud (NDCG@10)

Se evalúa la calidad del ranking de búsqueda mediante **NDCG@10** sobre todos los embeddings indexados, dejando out la imagen consultada en cada query.

| Métrica | Valor |
|---|---|
| NDCG@10 promedio | **0,9300** |
| NDCG@10 mediana | 0,9596 |
| Desvío estándar | 0,0792 |
| Mínimo | 0,6663 (Labradoodle) |
| Máximo | 1,0000 (Bermaise, Chow, Dalmation) |

**Interpretación.** Con un NDCG@10 promedio de ~0,93 y la mayoría de las razas por encima de 0,95, el sistema de búsqueda por similitud funciona muy bien usando **embeddings de ConvNeXt-Tiny pre-entrenado en ImageNet sin necesidad de fine-tuning**. Esto confirma que las features aprendidas en ImageNet ya capturan la variabilidad visual relevante para distinguir razas.

**10 peores razas:** Labradoodle (0,67), Bulldog (0,67), Lhasa (0,71), Cockapoo (0,74), Shih-Tzu (0,77), American Spaniel (0,80), Mex Hairless (0,82), Boston Terrier (0,83), Malinois (0,83), Border Collie (0,84). El patrón es claro: las razas **mixtas o muy parecidas a otras** (Labradoodle ≈ Labrador + Poodle; Cockapoo ≈ Cocker + Poodle; Lhasa ≈ Shih-Tzu) son las que más se confunden, lo cual es esperable y razonable.

### 6.2 Etapa 2 — Clasificación supervisada (test = 700 imágenes)

| Métrica | ResNet18 FT | CNN custom |
|---|---:|---:|
| Accuracy | **0,9543** | 0,8700 |
| Precision (macro) | **0,9588** | 0,8782 |
| Recall (macro) | **0,9543** | 0,8700 |
| Specificity | **0,9993** | 0,9981 |
| F1 (macro) | **0,9539** | 0,8696 |
| Errores totales | **32** | 91 |

Las matrices de confusión completas y las curvas de entrenamiento están en `etapa2_colab.ipynb`. ResNet18 mantiene F1 alto y parejo entre clases: sus peores razas siguen estando en 0,80-0,89 (Bulldog, Boston Terrier, American Spaniel, Shih-Tzu, Malinois, German Shepherd). La CNN custom tiene peores clases en 0,55-0,60 (Malinois, Labrador).

### 6.3 Evaluación sobre dataset propio (Acc = 0,8714 / ResNet18 FT)

Sobre las 70 imágenes del dataset propio (7 razas × 10), el ResNet18 fine-tuneado acierta 61 de 70 imágenes. El reporte por clase muestra que los errores son **razonables visualmente**: Border Collie → Collie (razas muy parecidas, pelaje blanco y negro y forma del rostro similar); Boxer ↔ Bulldog (estructura robusta y hocico corto similares); Chihuahua → Pomeranian con 87 % de confianza (perro chico, pelaje claro y abundante). Algunos errores muestran que el modelo se guía por **rasgos superficiales** (tamaño, color de pelaje) más que por características específicas de raza.

### 6.4 Etapa 3 — Pipeline end-to-end

Sobre las imágenes del dataset propio, YOLO11s detecta correctamente al perro en todos los casos con confianza ≥ 0,80 (típicamente ~0,90-0,95), incluso en escenas con fondos complejos (sillones, telas, otros objetos). Posteriormente, ResNet18 clasifica el recorte con la misma calidad que en la Etapa 2.B, manteniendo la accuracy ~0,87 (no degrada porque `SquarePad` preserva las proporciones).

## 7. Comparación entre enfoques

### 7.1 Búsqueda por similitud vs. clasificación supervisada

| Aspecto | Búsqueda por similitud (Etapa 1) | Clasificación supervisada (Etapa 2.A) |
|---|---|---|
| Entrenamiento | No requiere fine-tuning, solo indexar | Requiere fine-tuning con labels |
| Inferencia | Embedding + búsqueda k-NN (CPU OK) | Forward pass de ResNet18 (más rápido) |
| Métrica natural | NDCG@10 (ranking) | Accuracy/F1 (decisión dura) |
| Resultado | NDCG@10 ≈ 0,93 | Accuracy ≈ 0,954 |
| Explicabilidad | Alta: muestra los 10 vecinos | Baja: caja negra |
| Extensible a clases nuevas | Sí, solo agregar embeddings | No, hay que reentrenar |
| Riesgo de overfit | Bajo | Mayor (con dataset chico) |

**Conclusión.** Para 70 razas con datos suficientes, la **clasificación supervisada gana en accuracy**, pero la **búsqueda por similitud es más versátil**: permite mostrar al usuario imágenes parecidas (no solo una etiqueta) y agregar razas nuevas sin reentrenar. Ambos enfoques son complementarios y la app los expone como pestañas independientes.

### 7.2 ResNet18 fine-tuned vs. CNN custom

Sobre el test set de 700 imágenes:

| Categoría | Cantidad | Porcentaje |
|---|---:|---:|
| Aciertan ambos modelos | 595 | 85,00 % |
| Falla solo ResNet18 | 14 | 2,00 % |
| Falla solo CNN custom | 73 | 10,43 % |
| Fallan ambos | 18 | 2,57 % |

**ResNet18 fine-tuneado es claramente superior**: 32 errores vs 91 (casi el triple). El F1-macro confirma la ventaja (0,954 vs 0,870): no solo acierta más en general, sino que el rendimiento es más parejo entre clases. Las peores clases de ResNet18 están en 0,80-0,89; las de la CNN custom bajan hasta 0,55-0,60.

**Por qué.** ResNet18 parte de pesos pre-entrenados en ImageNet, por lo que ya posee filtros capaces de detectar bordes, texturas, formas, patrones de pelaje y estructuras visuales generales. La CNN custom debe aprender esas representaciones **desde cero** con un dataset relativamente limitado para 70 clases (200 imágenes por clase, varias sintéticas), lo cual la deja en desventaja.

La CNN custom logra un rendimiento aceptable y serviría como baseline en un entorno sin acceso a modelos pre-entrenados, pero no justifica el mayor costo computacional (más épocas, más parámetros entrenados desde cero) cuando ResNet18 fine-tuned está disponible. 

## 8. Problemas encontrados y soluciones implementadas

### 8.1 Desbalance moderado del split de train (ratio 3,05×)

**Problema.** Shih-Tzu tenía 198 imágenes mientras que American Hairless solo 65, lo cual sesgaría el entrenamiento hacia las razas mayoritarias.

**Solución.** Se implementó `scripts/balance_train_dataset.py`, que hace **oversampling con data augmentation** vía `albumentations` hasta llevar todas las clases a 200 imágenes, sin modificar el dataset original.

### 8.2 Imágenes borrosas en el dataset

**Problema.** Una pequeña porción de las imágenes de train estaban desenfocadas, lo que podía contaminar tanto los embeddings como el entrenamiento del clasificador.

**Solución.** Filtrado por **varianza del Laplaciano** con umbral conservador en 30. Se descartaron 5 imágenes (0,06 %), movidas a `data/dataset/train_descartadas`.

### 8.3 Nombre de carpeta con espacios duplicados

**Problema.** En el split `valid` había una carpeta llamada `"American  Spaniel"` (dos espacios) que no matcheaba con `"American Spaniel"` del resto de los splits, lo que provocaba un desfasaje al usar `ImageFolder`.

**Solución.** Se renombró manualmente la carpeta (`mv "American  Spaniel" "American Spaniel"`).

### 8.4 Desfasaje de índices entre dataset propio y train

**Problema.** `ImageFolder` asigna índices alfabéticamente por carpeta. Como el dataset propio tiene solo 7 razas, los índices locales no coincidían con los globales del modelo (70 clases), y predicciones correctas aparecían como erradas.

**Solución.** Se construye `own_idx_to_train_idx` mapeando cada índice local del dataset propio al índice global del `class_to_idx` usado durante el entrenamiento, y se pasa como `target_transform` al `ImageFolder` del dataset propio.

### 8.5 Recortes rectangulares de YOLO degradaban la clasificación

**Problema.** Los bounding boxes de YOLO tienen aspect ratio variable (un perro parado es alto, uno acostado es ancho). Si se aplica `Resize((224, 224))` directamente, las proporciones del animal se deforman y la clasificación sufre.

**Solución.** Se implementó `SquarePad` (transform custom) que rellena con negro hasta volver la imagen cuadrada **antes** del resize, preservando las proporciones del perro. Esto se aplica solo en el pipeline de la Etapa 3 (`classify_detected_dog`); en la Etapa 2 las imágenes ya vienen cuadradas o casi cuadradas.

## 9. Modificaciones fuera de las funciones indicadas

Las funciones que se pidieron implementar son: `extract_embedding`, `search_similar_images`, `predict_breed_from_neighbors`, `train_classifier`, `evaluate_classifier`, `detect_dogs` y `classify_detected_dog`. Todas se implementaron en los servicios provistos (`similarity_service.py`, `classifier_service.py`, `detection_service.py`) sin modificar la orquestación del repositorio base. Las modificaciones realizadas **fuera** de esas funciones se documentan a continuación:

1. **`scripts/balance_train_dataset.py` (script nuevo).**: El split de train tenía un ratio de desbalance de 3,05× que sesgaría el entrenamiento. El script genera una **copia** balanceada en `data/dataset_balanced/train` sin modificar el dataset original. Necesario para que el clasificador de la Etapa 2 no se sesgue hacia las razas mayoritarias.

2. **Clase `SquarePad` en `detection_service.py`.**: Los recortes rectangulares de YOLO se deformarían al hacer `Resize((224, 224))`. La clase rellena con negro antes del resize para preservar la proporción y queda **dentro** del preprocesamiento usado por `classify_detected_dog`.

3. **Métodos privados de entrenamiento (`_train_resnet18`, `_train_cnn_custom`, `_run_training_epoch`, `_run_validation_epoch`, `_save_cnn_custom_checkpoint`) agregados en `classifier_service.py`.** : descomponer `train_classifier()`. La interfaz pública (`train_classifier`, `evaluate_classifier`) se mantiene exactamente como pide la consigna.

Ninguna de las modificaciones cambia la interfaz pública de los servicios ni la forma en que se invocan desde la aplicación Gradio o desde los tests, y todas son necesarias para que el sistema funcione correctamente al levantarlo con `docker compose up`.